#Music Generation with AI




## STEP 1: CONNECT GOOGLE DRIVE


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

os.listdir('/content/drive/MyDrive/')

['Classroom',
 'Lab 1.gdoc',
 'Lab 2.gdoc',
 'Paul Deitel, Harvey Deitel - C How to Program-Pearson (2022).gdoc',
 'Lecture 3-SelectionStatements (Part 1).gslides',
 'Lecture 4-SelectionStatements (Part 2).gslides',
 'Lab assignment1.gdoc',
 'Lab 4.gdoc',
 'Lecture 6 - Nested Loops & Input Validation Loop_m.gslides',
 'LabAssignment2_B.gdoc',
 'LAB EXA 1.xlsx',
 'WORDEXAM.docx',
 'SP24-BCS-XXX (1).gdoc',
 'SP24-BCS-XXX.gdoc',
 'Lab 6 (1).gdoc',
 'Lab 6.gdoc',
 'Lecture 11 - Introduction to Arrays.gslides',
 'Lecture 12 - Introduction to Pointers.gslides',
 'Lecture 13 - Applications of pointers(Input and output arguments).gslides',
 'Lecture 15 - Array with Functions and poitners-2.gslides',
 'Lab Task - Arrays (1).gdoc',
 'Lab Task - Arrays.gdoc',
 'Lecture 16 - Linear Search _ Bubble Sort.gslides',
 'Lecture 17 - Introduction to 2D arrays.gslides',
 'Lecture 18 - Passing 2D arrays, partially-filled arrays to functions.gslides',
 'Lab Assignment 4 (1).gdoc',
 'Lab Assignment 4.gdoc',


In [8]:
import zipfile

zip_path = "/content/drive/MyDrive/maestro-v3.0.0-midi.zip"

extract_path = "/content/drive/MyDrive/maestro_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [9]:
import os

os.listdir('/content/drive/MyDrive/maestro_dataset')[:10]

['maestro-v3.0.0']

In [10]:
import glob

midi_files = glob.glob(
    "/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/**/*.midi",
    recursive=True
)

print("Total MIDI files found:", len(midi_files))

print(midi_files[:5])

Total MIDI files found: 1276
['/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi', '/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_09_R1_2004_05_ORIG_MID--AUDIO_09_R1_2004_06_Track06_wav.midi', '/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_14_R1_2004_01-03_ORIG_MID--AUDIO_14_R1_2004_01_Track01_wav.midi', '/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_01_R1_2004_01-02_ORIG_MID--AUDIO_01_R1_2004_03_Track03_wav.midi', '/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_13_01_2004_01-05_ORIG_MID--AUDIO_13_R1_2004_09_Track09_wav.midi']


##STEP 2: INSTALL LIBRARIES

In [11]:
!pip install music21
!pip install pretty_midi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 72.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.9 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=48719244ac681574d880845377cc32c18f0822107733bc8622ccb7f6f6178558
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


##STEP 3: IMPORT LIBRARIES

In [12]:
import glob
import numpy as np
import pickle

from music21 import converter, instrument, note, chord, stream

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import to_categorical

## STEP 6: LOAD MIDI FILES

In [13]:
midi_files = glob.glob(
    "/content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/**/*.midi",
    recursive=True
)

## STEP 7: EXTRACT NOTES FROM MIDI FILES

In [14]:
notes = []

from music21 import converter, instrument, note, chord

# Use only first 50 files for faster training
# You can increase later
for file in midi_files[:50]:

    print("Processing:", file)

    try:
        midi = converter.parse(file)

        notes_to_parse = None

        try:
            s2 = instrument.partitionByInstrument(midi)
            notes_to_parse = s2.parts[0].recurse()

        except:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:

            if isinstance(element, note.Note):
                notes.append(str(element.pitch))

            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    except:
        print("Skipped file:", file)

print("\nTotal notes extracted:", len(notes))

Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_08_R1_2004_01-02_ORIG_MID--AUDIO_08_R1_2004_01_Track01_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_09_R1_2004_05_ORIG_MID--AUDIO_09_R1_2004_06_Track06_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_14_R1_2004_01-03_ORIG_MID--AUDIO_14_R1_2004_01_Track01_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_01_R1_2004_01-02_ORIG_MID--AUDIO_01_R1_2004_03_Track03_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_SMF_13_01_2004_01-05_ORIG_MID--AUDIO_13_R1_2004_09_Track09_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/MIDI-Unprocessed_XP_18_R1_2004_01-02_ORIG_MID--AUDIO_18_R1_2004_03_Track03_wav.midi
Processing: /content/drive/MyDrive/maestro_dataset/maestro-v3.0.0/2004/M

## STEP 8: SAVE EXTRACTED NOTES

In [15]:
import pickle

with open('/content/drive/MyDrive/notes.pkl', 'wb') as filepath:
    pickle.dump(notes, filepath)

print("Notes saved successfully!")

Notes saved successfully!


## STEP 9: PREPARE TRAINING DATA

In [16]:
import numpy as np
from tensorflow.keras.utils import to_categorical

sequence_length = 100

# Get all unique notes
pitchnames = sorted(set(item for item in notes))

# Create dictionary
note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

network_input = []
network_output = []

for i in range(0, len(notes) - sequence_length):

    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[char] for char in sequence_in])

    network_output.append(note_to_int[sequence_out])

n_patterns = len(network_input)

print("Total Patterns:", n_patterns)

# Reshape input
network_input = np.reshape(
    network_input,
    (n_patterns, sequence_length, 1)
)

# Normalize
network_input = network_input / float(len(pitchnames))

# One-hot encode output
network_output = to_categorical(network_output)

Total Patterns: 154233


## STEP 10 : BUILS LSTM MODEL

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import BatchNormalization

model = Sequential()

model.add(LSTM(
    256,
    input_shape=(network_input.shape[1], network_input.shape[2]),
    return_sequences=True
))

model.add(Dropout(0.3))

model.add(LSTM(256))

model.add(Dense(256))

model.add(Dropout(0.3))

model.add(Dense(len(pitchnames), activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 256)       │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1493)           │       383,701 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,238,997 (4.73 MB)

 Trainable params: 1,238,997 (4.73 MB)

 Non-trainable params: 0 (0.00 B)

## STEP 11: TRAIN THE MODEL

In [18]:
history = model.fit(
    network_input,
    network_output,
    epochs=20,
    batch_size=64
)

Epoch 1/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 51s 20ms/step - loss: 5.4042
Epoch 2/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.2964
Epoch 3/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 51s 21ms/step - loss: 5.2785
Epoch 4/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 51s 21ms/step - loss: 5.2668
Epoch 5/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 82s 21ms/step - loss: 5.2464
Epoch 6/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.2217
Epoch 7/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.1935
Epoch 8/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.1564
Epoch 9/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.1112
Epoch 10/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.0690
Epoch 11/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 5.0248
Epoch 12/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 4.9749
Epoch 13/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 4.9235
Epoch 14/20
2410/2410 ━━━━━━━━━━━━━━━━━━━━ 50s 21ms/step - loss: 4.8769
E

## STEP 12: SAVE TRAINED MODEL

In [19]:
model.save('/content/drive/MyDrive/music_generation_model.h5')

print("Model saved successfully!")

Model saved successfully!


##STEP 13: GENERATE NEW MUSIC

In [20]:
start = np.random.randint(0, len(network_input)-1)

int_to_note = dict(
    (number, note) for number, note in enumerate(pitchnames)
)

pattern = network_input[start]

prediction_output = []

for note_index in range(200):

    prediction_input = np.reshape(
        pattern,
        (1, len(pattern), 1)
    )

    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(
        prediction_input,
        verbose=0
    )

    index = np.argmax(prediction)

    result = int_to_note[index]

    prediction_output.append(result)

    pattern = np.append(pattern, index)

    pattern = pattern[1:len(pattern)]

print("Music generated successfully!")

Music generated successfully!


## STEP 14: CONVERT GENERATED NOTES TO MIDI

In [21]:

from music21 import stream

offset = 0
output_notes = []

for pattern in prediction_output:

    if ('.' in pattern) or pattern.isdigit():

        notes_in_chord = pattern.split('.')
        notes_list = []

        for current_note in notes_in_chord:

            new_note = note.Note(int(current_note))
            new_note.storedInstrument = instrument.Piano()

            notes_list.append(new_note)

        new_chord = chord.Chord(notes_list)

        new_chord.offset = offset

        output_notes.append(new_chord)

    else:

        new_note = note.Note(pattern)

        new_note.offset = offset

        new_note.storedInstrument = instrument.Piano()

        output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)

midi_stream.write(
    'midi',
    fp='/content/drive/MyDrive/generated_music.mid'
)

print("Generated MIDI file saved!")

Generated MIDI file saved!
